# DataModeling Metrics Analysis

This notebook reproduces `show_results.py` outputs and adds richer metrics (including CAC reliability-related curves) from `experience_task.jsonl`.

In [ ]:
%matplotlib inline
import json
import ast
from pathlib import Path
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt


## Configuration

In [ ]:
def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for c in [cur] + list(cur.parents):
        if c.name == "DatawiseAgent-s4":
            return c
    for c in [cur] + list(cur.parents):
        candidate = c / "DatawiseAgent-s4"
        if candidate.exists() and candidate.is_dir():
            return candidate
    for c in [cur] + list(cur.parents):
        if (c / "evaluation").exists() and (c / "datawiseagent").exists() and (c / "configs").exists():
            return c
    return cur

project_root = find_project_root(Path.cwd())
model = "gpt-4o-mini"
run_name = "cac_llm"  # change to other run dirs if needed

results_path = project_root / "evaluation" / "experimental_results" / "DataModeling-DSBench" / run_name / "results.jsonl"
baseline_path = project_root / "evaluation" / "DataModeling" / "save_performance" / "baseline"
gt_path = project_root / "evaluation" / "DataModeling" / "save_performance" / "GT"
performances_dir = results_path.parent / "performances"
experience_path = results_path.parent / "experience_task.jsonl"

print(f"Project root: {project_root}")
print(f"Using model: {model}")
print(f"Results path: {results_path}")
print(f"Performances dir: {performances_dir}")
print(f"Experience path: {experience_path}")
print(f"GT path: {gt_path}")
print(f"Baseline path: {baseline_path}")



In [ ]:
def parse_metric_file(path: Path):
    text = path.read_text(encoding="utf-8").strip()
    try:
        return ast.literal_eval(text)
    except Exception:
        return text

def show_results_like(results_path: Path, baseline_path: Path, gt_path: Path, performances_dir: Path):
    results = [json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines() if line.strip()]

    task_complete = 0
    scores = []
    all_times = []

    for item in results:
        name = item["name"]
        all_times.append(item.get("time", 0))

        gt = parse_metric_file(gt_path / name / "result.txt")
        bl = parse_metric_file(baseline_path / name / "result.txt")

        pred_metric_path = performances_dir / name / "result.txt"
        if not pred_metric_path.exists():
            scores.append(0)
            continue

        pre = parse_metric_file(pred_metric_path)
        if str(pre).lower() == "nan":
            scores.append(0)
            continue

        try:
            sc = max(0, (float(pre) - float(bl)) / (float(gt) - float(bl)))
        except ZeroDivisionError:
            sc = 0
        scores.append(sc)
        task_complete += 1

    completion_rate = task_complete / len(scores) if scores else 0
    avg_time = sum(all_times) / len(all_times) if all_times else 0
    avg_performance = sum(scores) / len(scores) if scores else 0

    print(len(results))
    print(f"Task completion rate is {task_complete}/{len(scores)}={completion_rate}")
    print(f"The average time consuming is {avg_time}")
    print(f"The performance is {avg_performance} with {len(scores)} out of 74 tasks scored.")

    return {
        "task_count": len(results),
        "task_complete": task_complete,
        "completion_rate": completion_rate,
        "avg_time": avg_time,
        "avg_performance": avg_performance,
        "scores": scores,
        "times": all_times,
    }

def robust_convolve(data_list, window):
    if len(data_list) == 0:
        return np.array([])
    if window <= 1 or window > len(data_list):
        return np.array(data_list)
    return np.convolve(data_list, np.ones(window) / window, mode="valid")

def analyze_experience_metrics(experience_path: Path):
    if not experience_path.exists():
        print(f"experience file not found: {experience_path}")
        return None

    end_records = []
    role_router_calls = defaultdict(int)

    # CAC per-routing-step reconstructed metrics from agent records carrying `cac`
    # Key: (session_id, task_id, step_id, small_step_id)
    cac_groups = defaultdict(list)

    for line in experience_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)

        if str(rec.get("step_id")) == "end":
            end_records.append(rec)

        if rec.get("agent_path") == "ROUTER_CALL":
            role = rec.get("role") or "unknown"
            role_router_calls[role] += 1

        if rec.get("agent_path") != "ROUTER_CALL" and rec.get("cac"):
            key = (
                rec.get("session_id"),
                rec.get("task_id"),
                rec.get("step_id"),
                rec.get("small_step_id"),
            )
            cac_groups[key].append(rec.get("cac", {}))

    if not end_records:
        print("No end records found in experience_task.jsonl")
        return None

    def avg(key):
        vals = [float(r.get(key, 0)) for r in end_records]
        return sum(vals) / len(vals) if vals else 0

    summary = {
        "records": len(end_records),
        "avg_total_latency": avg("total_latency"),
        "avg_agent_latency": avg("agent_latency"),
        "avg_router_latency": avg("router_latency"),
        "avg_network_latency": avg("network_latency"),
        "avg_total_token_count": avg("total_token_count"),
        "avg_agent_token_count": avg("agent_token_count"),
        "avg_router_token_count": avg("router_token_count"),
        "avg_step_num": avg("step_num"),
        "avg_call_count": avg("call_count"),
        "avg_network_failure_rate": avg("network_failure_rate"),
        "avg_agent_failure_rate": avg("agent_failure_rate"),
        "role_router_calls": dict(role_router_calls),
    }

    # Reconstruct CAC reliability-like traces
    # node_reliability formula follows cac_router implementation:
    # p_safe_quality = max(0.1, Mean - 0.8*Std)
    # node_reliability = min(p_safe_quality * P_Success, 0.98)
    # R_sys_est = 1 - Π(1 - node_reliability)
    cac_trace = {
        "routing_index": [],
        "selected_agent_count": [],
        "avg_p_success": [],
        "r_sys_est": [],
    }

    sorted_keys = sorted(cac_groups.keys(), key=lambda x: (str(x[1]), str(x[2]), str(x[3])))
    for idx, key in enumerate(sorted_keys, 1):
        items = cac_groups[key]
        if not items:
            continue

        node_rels = []
        p_successes = []
        for c in items:
            mean = float(c.get("Mean", 0.0))
            std = float(c.get("Std", 0.0))
            p_success = float(c.get("P_Success", 0.0))
            p_safe_quality = max(0.1, mean - 0.8 * std)
            node_rel = min(p_safe_quality * p_success, 0.98)
            node_rels.append(node_rel)
            p_successes.append(p_success)

        joint_failure = 1.0
        for nr in node_rels:
            joint_failure *= (1.0 - nr)
        r_sys = 1.0 - joint_failure

        cac_trace["routing_index"].append(idx)
        cac_trace["selected_agent_count"].append(len(items))
        cac_trace["avg_p_success"].append(sum(p_successes) / len(p_successes))
        cac_trace["r_sys_est"].append(r_sys)

    print("\n--- Extra Metrics (from experience_task.jsonl) ---")
    for k, v in summary.items():
        print(f"{k}: {v}")
    if cac_trace["routing_index"]:
        print("CAC reliability trace available:", len(cac_trace["routing_index"]), "routing steps")
    else:
        print("CAC reliability trace not found (no `cac` fields in agent records).")

    return end_records, summary, cac_trace



In [ ]:
assert results_path.exists(), f"results_path not found: {results_path}"
assert baseline_path.exists(), f"baseline_path not found: {baseline_path}"
assert gt_path.exists(), f"gt_path not found: {gt_path}"

show_results_metrics = show_results_like(results_path, baseline_path, gt_path, performances_dir)
experience_data = analyze_experience_metrics(experience_path)



In [ ]:
rolling_window = 5
figsize = (12, 4)

if experience_data is not None:
    end_records, summary, cac_trace = experience_data
    task_indices = np.arange(1, len(end_records) + 1)

    def get_series(key):
        return [float(r.get(key, 0)) for r in end_records]

    metrics = {
        "avg_planner_call_count": get_series("avg_planner_call_count"),
        "step_num": get_series("step_num"),
        "call_count": get_series("call_count"),
        "agent_failure_rate": get_series("agent_failure_rate"),
        "network_failure_rate": get_series("network_failure_rate"),
        "agent_token_count": get_series("agent_token_count"),
        "router_token_count": get_series("router_token_count"),
        "total_token_count": get_series("total_token_count"),
        "agent_latency": get_series("agent_latency"),
        "router_latency": get_series("router_latency"),
        "network_latency": get_series("network_latency"),
        "total_latency": get_series("total_latency"),
    }

    w = min(rolling_window, len(task_indices)) if len(task_indices) > 0 else 1
    w = max(w, 1)
    x = task_indices[w-1:] if w > 1 else task_indices

    # 1) call counts / steps
    plt.figure(figsize=figsize)
    plt.plot(x, robust_convolve(metrics["avg_planner_call_count"], w), label="Avg Planner Call Count")
    plt.plot(x, robust_convolve(metrics["step_num"], w), label="Step Num")
    plt.plot(x, robust_convolve(metrics["call_count"], w), label="Total Call Count")
    plt.title("Call Counts & Step Numbers")
    plt.xlabel("Task Index")
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

    # 2) failure rates
    plt.figure(figsize=figsize)
    plt.plot(x, robust_convolve(metrics["agent_failure_rate"], w), label="Agent Failure Rate")
    plt.plot(x, robust_convolve(metrics["network_failure_rate"], w), label="Network Failure Rate")
    plt.title("Failure Rates")
    plt.xlabel("Task Index")
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

    # 3) token counts
    plt.figure(figsize=figsize)
    plt.plot(x, robust_convolve(metrics["agent_token_count"], w), label="Agent Token Count")
    plt.plot(x, robust_convolve(metrics["router_token_count"], w), label="Router Token Count")
    plt.plot(x, robust_convolve(metrics["total_token_count"], w), label="Total Token Count")
    plt.title("Token Counts")
    plt.xlabel("Task Index")
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

    # 4) latency
    plt.figure(figsize=figsize)
    plt.plot(x, robust_convolve(metrics["agent_latency"], w), label="Agent Latency")
    plt.plot(x, robust_convolve(metrics["router_latency"], w), label="Router Latency")
    plt.plot(x, robust_convolve(metrics["network_latency"], w), label="Network Latency")
    plt.plot(x, robust_convolve(metrics["total_latency"], w), label="Total Latency")
    plt.title("Latency Metrics")
    plt.xlabel("Task Index")
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

    # 5) CAC reliability-style curve (from `cac` fields)
    if cac_trace["routing_index"]:
        rx = np.array(cac_trace["routing_index"])
        rw = min(rolling_window, len(rx)) if len(rx) > 0 else 1
        rw = max(rw, 1)
        rxx = rx[rw-1:] if rw > 1 else rx

        plt.figure(figsize=figsize)
        plt.plot(rxx, robust_convolve(cac_trace["r_sys_est"], rw), label="R_sys_est (reconstructed)")
        plt.axhline(y=0.95, color="r", linestyle="--", label="Target 0.95")
        plt.ylim(0, 1.02)
        plt.title("CAC Reliability Curve (R* proxy)")
        plt.xlabel("Routing Step Index")
        plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

        plt.figure(figsize=figsize)
        plt.plot(rxx, robust_convolve(cac_trace["selected_agent_count"], rw), label="Selected Agent Count")
        plt.plot(rxx, robust_convolve(cac_trace["avg_p_success"], rw), label="Avg P_Success")
        plt.title("CAC Selection Dynamics")
        plt.xlabel("Routing Step Index")
        plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
    else:
        print("No CAC trace detected; skip CAC reliability plots.")



Notes:
- `R_sys_est` is reconstructed from logged `cac` fields using the same node reliability formula in `cac_router.py`.
- It is a practical proxy for CAC reliability behavior from logs.